# 0001 — Equal-weight buy-and-hold

| | |
|---|---|
| **Hypothesis** | None. This is the baseline every later backtest is compared against, and the first time the position → value → equity → metrics mechanics are written. |
| **Universe** | ITUB4, PETR4, SUZB3, VALE3 (equal weight). Benchmark: BOVA11. |
| **Period** | 2019-01-01 → 2025-12-31, daily, adjusted prices. |
| **Result** | _Fill in after running._ |
| **Conclusion** | _Fill in after running._ |
| **Drove in `src/`** | _Fill in: what this notebook showed is missing from the library._ |

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from ibexQuant.data.calendars.calendar import align_to_calendar
from ibexQuant.data.calendars.pandas_market_calendars.calendar import (
    PandasMarketCalendarsCalendar,
)
from ibexQuant.data.ingestion.validation import check_bars
from ibexQuant.data.ingestion.yfinance.provider import YFinanceProvider
from ibexQuant.data.panel import Panel
from ibexQuant.features.kernels import simple_returns

In [ ]:
PORTFOLIO_SYMBOLS = ("ITUB4.SA", "PETR4.SA", "SUZB3.SA", "VALE3.SA")
BENCHMARK_SYMBOL = "BOVA11.SA"
START = "2019-01-01"
END = "2025-12-31"

INITIAL_CAPITAL = 100_000.0
SESSIONS_PER_YEAR = 252

pd.set_option("display.precision", 4)

## Data

Standard pipeline: `get_bars → check_bars → align_to_calendar → Panel`. The benchmark
is downloaded together with the portfolio so both share the same calendar-aligned index.

In [ ]:
provider = YFinanceProvider(frequency="1d")
bars = provider.get_bars(
    [*PORTFOLIO_SYMBOLS, BENCHMARK_SYMBOL], start=START, end=END, adjusted=True
)

issues = check_bars(bars)
flagged = issues[issues.any(axis=1)]
print(f"check_bars: {len(flagged)} row(s) flagged")
if not flagged.empty:
    display(flagged)

aligned = align_to_calendar(bars, calendar=PandasMarketCalendarsCalendar("B3"))
panel = Panel.from_bars(aligned)
close = panel.field("close")
panel

In [ ]:
# First observation and real gaps per symbol: a late first date moves the backtest start.
pd.DataFrame(
    {
        "first_observation": close.apply(pd.Series.first_valid_index),
        "missing_sessions": close.isna().sum(),
    }
)

## Mechanics

Capital is split by weight at the **close of the first session where every symbol has a
price**, converted to a number of shares, and never traded again. Weights therefore drift
with prices — that drift is what distinguishes buy-and-hold from a rebalanced portfolio.

There is no signal, so there is no look-ahead: the only decision is at `t0` and it uses
only the price at `t0`.

During a gap, a holding is marked at its last traded price. This is valuation, not a
feature: the position still exists while the asset is suspended, so the forward-fill does
not conflict with the observation-time policy (ADR 0003).

In [ ]:
def buy_and_hold(close: pd.DataFrame, weights: pd.Series, capital: float) -> pd.DataFrame:
    """Value of each holding, bought at the first common session and never rebalanced."""
    if not np.isclose(weights.sum(), 1.0):
        raise ValueError(f"Weights must sum to 1, got {weights.sum():.6f}.")

    prices = close[weights.index]
    fully_observed = prices.notna().all(axis=1)
    if not fully_observed.any():
        raise ValueError("No session in which every symbol has a price.")

    marked = prices.loc[fully_observed.idxmax() :].ffill()
    shares = capital * weights / marked.iloc[0]
    return marked * shares

In [ ]:
portfolio_weights = pd.Series(1.0 / len(PORTFOLIO_SYMBOLS), index=list(PORTFOLIO_SYMBOLS))
holdings = buy_and_hold(close, portfolio_weights, INITIAL_CAPITAL)

# The benchmark starts on the portfolio's first session so both curves share t0.
benchmark_holdings = buy_and_hold(
    close.loc[holdings.index[0] :], pd.Series({BENCHMARK_SYMBOL: 1.0}), INITIAL_CAPITAL
)

equity = pd.DataFrame(
    {
        "portfolio": holdings.sum(axis=1),
        "benchmark": benchmark_holdings.sum(axis=1),
    }
)
print(f"backtest: {equity.index[0].date()} -> {equity.index[-1].date()}, {len(equity)} sessions")
equity.tail()

## Metrics

In [ ]:
def drawdown(equity: pd.DataFrame) -> pd.DataFrame:
    """Relative distance of each equity curve from its running peak."""
    return equity / equity.cummax() - 1.0


def performance_summary(equity: pd.DataFrame) -> pd.DataFrame:
    """One row per equity curve with return, risk and drawdown metrics."""
    returns = simple_returns(equity)
    years = (len(equity) - 1) / SESSIONS_PER_YEAR
    total_return = equity.iloc[-1] / equity.iloc[0] - 1.0
    volatility = returns.std() * np.sqrt(SESSIONS_PER_YEAR)
    curve_drawdown = drawdown(equity)

    return pd.DataFrame(
        {
            "total_return": total_return,
            "cagr": (1.0 + total_return) ** (1.0 / years) - 1.0,
            "volatility": volatility,
            # Zero risk-free rate: overstates Sharpe in Brazil, where the CDI is far from zero.
            "sharpe_rf0": returns.mean() * SESSIONS_PER_YEAR / volatility,
            "max_drawdown": curve_drawdown.min(),
            "max_drawdown_date": curve_drawdown.idxmin(),
        }
    )

In [ ]:
performance_summary(equity)

In [ ]:
# Contribution of each holding to the final result.
pd.DataFrame(
    {
        "final_value": holdings.iloc[-1],
        "total_return": holdings.iloc[-1] / holdings.iloc[0] - 1.0,
        "final_weight": holdings.iloc[-1] / holdings.iloc[-1].sum(),
    }
)

## Charts

In [ ]:
fig, (ax_equity, ax_drawdown) = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True, height_ratios=[2, 1]
)

(equity / equity.iloc[0]).plot(ax=ax_equity, title="Growth of 1")
ax_equity.set_ylabel("multiple of initial capital")
ax_equity.grid(alpha=0.3)

drawdown(equity).plot(ax=ax_drawdown, title="Drawdown", legend=False)
ax_drawdown.set_ylabel("drawdown")
ax_drawdown.grid(alpha=0.3)

fig.tight_layout()

In [ ]:
weights_over_time = holdings.div(holdings.sum(axis=1), axis=0)

ax = weights_over_time.plot.area(figsize=(12, 4), title="Weight drift (never rebalanced)")
ax.set_ylabel("weight")
ax.set_ylim(0, 1)
ax.legend(loc="upper left")
plt.tight_layout()

## Known simplifications

- **No costs.** A single purchase makes this negligible here, but not for any strategy
  that trades.
- **Zero risk-free rate.** Sharpe ignores the CDI, which strongly flatters any Brazilian
  equity strategy.
- **Survivorship bias.** The universe is today's blue chips, chosen with hindsight.
- **Adjusted prices.** Dividends are reinvested inside each asset, which is what yfinance's
  adjustment implies; there is no cash account.
- **Fractional shares and no lot size.** B3 trades in lots of 100 (or the fractional
  market), ignored here.

## Friction observed → candidates for `src/`

_Fill in while working: anything repeated here that the next notebooks will also need._

- Equity curve from weights (`buy_and_hold`) and performance metrics — annualization lives
  outside kernels (ADR 0008), so these need their own home.
- A risk-free series (CDI) for excess returns.
- Downloads repeated on every run — the `CachingProvider` already anticipated in the
  provider's docstring.